In [3]:
!nvidia-smi
import torch

print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB"
    )

Thu Jun 11 17:41:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1: Clone Repository and Setup Environment

In [4]:
import os
import getpass
from pathlib import Path

# Configuration
GITHUB_USER = "sattary"
REPO_NAME = "ali_proj"
BRANCH = "fix-review"  # Change if using different branch
PROJECT_DIR = "ali_proj"

print("Enter your GitHub Personal Access Token (PAT):")
PAT = getpass.getpass()
REPO_URL = f"https://{PAT}@github.com/{GITHUB_USER}/{REPO_NAME}.git"

# 1. Clone Repository
if not Path(PROJECT_DIR).exists():
    print(f"Cloning {REPO_NAME} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
else:
    print("Repository already cloned. Pulling latest changes...")
    !cd {PROJECT_DIR} && git pull origin {BRANCH}

%cd {PROJECT_DIR}

# 2. Install uv
print("\nInstalling uv...")
!pip install -q uv

# 3. Set MPLBACKEND for Kaggle compatibility
os.environ['MPLBACKEND'] = 'Agg'
print("\nSet MPLBACKEND=Agg for headless environments")

# 4. Sync Dependencies
print("\nSyncing dependencies...")
!uv sync

print("\n✓ Setup complete!")

Enter your GitHub Personal Access Token (PAT):


Cloning ali_proj (branch: fix-review)...
Cloning into 'ali_proj'...
remote: Enumerating objects: 1166, done.
remote: Counting objects: 100% (475/475), done.
remote: Compressing objects: 100% (260/260), done.
remote: Total 1166 (delta 282), reused 381 (delta 199), pack-reused 691 (from 1)
Receiving objects: 100% (1166/1166), 39.73 MiB | 19.95 MiB/s, done.
Resolving deltas: 100% (667/667), done.
Filtering content: 100% (13/13), 2.25 MiB | 1.02 MiB/s, done.
/content/ali_proj

Installing uv...

Set MPLBACKEND=Agg for headless environments

Syncing dependencies...
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 76 packages in 1ms
Prepared 74 packages in 1m 27s                                           
Installed 74 packages in 822ms                              
 + alembic==1.18.4
 + ali-proj==0.1.0 (from file:///content/ali_proj)
 + annotated-doc==0.0.4
 + click==8.3.1
 + colorlog==6.10.1
 + contourpy==1.3.3
 + cuda-bindings==12.9.4
 +

In [6]:
!uv pip install ipywidgets

Using Python 3.12.13 environment at: /usr
Resolved 81 packages in 335ms                                        
Prepared 1 package in 703ms                                              
Installed 1 package in 106ms                                
 + jedi==0.20.0


In [7]:
# Verify Git Repository
from pathlib import Path
import subprocess

# Check if we're in a git repo
result = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True
)
if result.returncode == 0:
    repo_root = result.stdout.strip()
    print(f"✓ Git repository found at: {repo_root}")
    print(f"✓ Current directory: {Path.cwd()}")
else:
    print("Error: Not in a git repository!")
    print(f"Current directory: {Path.cwd()}")
    raise RuntimeError("Git repository not found")

# Show repo status
!git status

✓ Git repository found at: /content/ali_proj
✓ Current directory: /content/ali_proj
On branch fix-review
Your branch is up to date with 'origin/fix-review'.

nothing to commit, working tree clean


## Phase 1: Deterministic Generation (`generate`)

Generate the synthetic interferogram dataset to HDF5 shards. The cryptographic seed guarantees mathematically invariant noise topologies.

In [8]:
# Phase 1: Generate Data
!uv run phase-unwrap generate \
    --num-samples 10000 \
    --shard-size 1000 \
    --out-dir data/kaggle_full \
    --seed 1337

print("\n✓ Data generation complete!")

Shards: 100% 180/180 [32:21<00:00, 10.78s/it]
Generated 180000 samples across 180 shards in data/kaggle_full
Saved data config: data/kaggle_full/data_config.yaml

✓ Data generation complete!


## Phase 2: Hyperparameter Optimization (`tune`)

Use Optuna's Bayesian TPE algorithm to isolate the absolute lowest-error configuration. The best configuration is automatically saved to `runs/optuna/best_config.yaml`.

In [17]:
!git pull

Already up to date.


In [ ]:
# Phase 2: Optuna Tune
# Automatically parallelizes across available GPUs
!uv run phase-unwrap tune \
    --use-amp \
    --n-trials 30 \
    --tune-epochs 15 \
    --study-name kaggle_10k_hpo \
    --n-workers 2 \
    --batch-size 16 \
    --data-dir data/kaggle_full/

print("\n✓ Tuning complete!")
print("Best config frozen to: runs/optuna/best_config.yaml")

[I 2026-06-11 18:38:16,651] A new study created in RDB with name: colab_180k_hpo
Running 30 trials (0 existing, 30 target)
  Trial 0 | epoch 1/15 | val MAE=1.8952 (best=1.8952)                  
  Trial 0 | epoch 2/15 | val MAE=3.5722 (best=1.8952)                
  Trial 0 | epoch 3/15 | val MAE=1.0715 (best=1.0715)                
Trial 0 Ep 4/15:  82% 7357/9000 [12:29<02:57,  9.23it/s, loss=1.4533]

: 

: 

Trial 0 Ep 4/15:  82% 7362/9000 [12:30<03:00,  9.08it/s, loss=1.4532]

## Phase 3: Primary Training and Evaluation (`train`)

Train the baseline network to convergence using the frozen `best_config.yaml`.

In [ ]:
# Phase 3: Train Primary Network
import os
config_arg = "--config runs/optuna/best_config.yaml" if os.path.exists("runs/optuna/best_config.yaml") else ""

!uv run phase-unwrap train \
    --use-amp \
    {config_arg} \
    --run-name exp_primary \
    --multi-gpu \
    --data-dir data/kaggle_full/

print("\n✓ Primary training complete!")

## Phase 4: Statistical Validation (`multiseed`)

Defend against 'lucky seed' anomalies. Spawns completely independent training convergences using the same frozen configuration, and automatically aggregates the metrics into Mean ± Std.

In [ ]:
# Phase 4: Statistical Validation
import os
config_arg = "--config runs/optuna/best_config.yaml" if os.path.exists("runs/optuna/best_config.yaml") else ""

!uv run phase-unwrap multiseed \
    --use-amp \
    {config_arg} \
    --run-name exp_multiseed \
    --num-seeds 3 \
    --multi-gpu \
    --data-dir data/kaggle_full/

print("\n✓ Multiseed validation complete!")

## Phase 5: Architectural Ablation (`ablation`)

Mathematically prove the necessity of your custom topology by systematically crippling the network. Exports a rigorous LaTeX comparison table.

In [ ]:
# Phase 5: Architectural Ablation
import os
config_arg = "--config runs/optuna/best_config.yaml" if os.path.exists("runs/optuna/best_config.yaml") else ""

!uv run phase-unwrap ablation \
    --use-amp \
    {config_arg} \
    --out-table results/tables/ablation.tex \
    --multi-gpu \
    --data-dir data/kaggle_full/

print("\n✓ Ablation study complete!")

## Step 6: Zip and Download Results

Run this cell to zip the `runs/` and `results/` directories so you can render them on your local machine.

In [ ]:
import shutil
from IPython.display import FileLink

print("Zipping runs and results...")
shutil.make_archive('training_results', 'zip', 'runs/')
shutil.make_archive('tables_results', 'zip', 'results/')
print("✓ Done!")
display(FileLink('training_results.zip'))
display(FileLink('tables_results.zip'))